# Treinamento do Modelo Preditivo de F1 (XGBoost)

Este notebook carrega os dados processados da camada Gold, treina um modelo XGBoost para prever a posição final e avalia sua performance.

## 1. Setup e Importações

In [ ]:
%pip install pandas numpy xgboost scikit-learn matplotlib seaborn python-dotenv boto3 s3fs pyarrow joblib

import os
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.metrics import mean_absolute_error, mean_squared_error
from dotenv import load_dotenv

load_dotenv()
BUCKET = os.environ.get('S3_BUCKET_NAME')

pd.set_option('display.max_columns', None)
sns.set_theme(style="whitegrid")

## 2. Carregamento dos Dados
Lendo os arquivos Parquet gerados no notebook de `feature_engineering_and_split`.

In [ ]:
try:
    print("Carregando dados de treino...")
    train_df = pd.read_parquet(f"s3://{BUCKET}/gold/train_data.parquet")
    print(f"Treino: {train_df.shape}")
    
    print("Carregando dados de teste...")
    test_df = pd.read_parquet(f"s3://{BUCKET}/gold/test_data.parquet")
    print(f"Teste: {test_df.shape}")
    
except Exception as e:
    print(f"Erro ao carregar dados: {e}")
    # Stop execution if load fails
    raise e

## 3. Preparação dos Dados (X e y)
Separando as variáveis preditoras (features) da variável alvo (target). Também removemos colunas que não devem ser usadas no treino (como IDs e datas).

In [ ]:
# Variável Alvo
TARGET = 'final_position'

# Colunas para remover do treino (IDs, vazamento de futuro, ou target)
drop_cols = [
    TARGET, 
    'meeting_key', 'session_key', 'driver_number', 
    'date_start', 'date_end', 
    'country_name', 'session_name', 'session_type', # Textos que não foram encoded (se houver)
    'year'
]

# Garantir que só removemos o que existe
features = [c for c in train_df.columns if c not in drop_cols]

# Selecionar apenas colunas numéricas para o XGBoost (ele lida bem, mas garante que strings soltas não quebrem)
# O One-Hot Encoding já deve ter transformado categorias em números.
X_train = train_df[features].select_dtypes(include=[np.number])
y_train = train_df[TARGET]

X_test = test_df[features].select_dtypes(include=[np.number])
y_test = test_df[TARGET]

# Alinhar colunas (garantir que teste tenha as mesmas do treino)
X_test = X_test[X_train.columns]

print(f"Features selecionadas ({len(X_train.columns)}): {list(X_train.columns)[:10]} ...")

## 4. Treinamento do Modelo (XGBoost)
Configurando e treinando um `XGBRegressor`. Usamos regressão porque a posição é ordinal.

In [ ]:
model = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=1000,
    learning_rate=0.05, # Taxa de aprendizado baixa para generalizar melhor
    max_depth=5,        # Profundidade moderada para evitar overfitting
    subsample=0.8,      # Amostra de linhas
    colsample_bytree=0.8, # Amostra de colunas
    random_state=42,
    early_stopping_rounds=50, # Para se não melhorar no teste
    n_jobs=-1
)

print("Iniciando treinamento...")
model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=100  # Log a cada 100 iterações
)
print("Treinamento concluído!")

## 5. Avaliação do Modelo
Analisando os erros nas previsões.

In [ ]:
# Previsões
y_pred = model.predict(X_test)

# Garantir que previsões não sejam negativas ou irreais
y_pred = np.clip(y_pred, 1, 20) 

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"\n--- Resultados no Teste (2025/Fallback) ---")
print(f"MAE (Erro Médio Absoluto): {mae:.2f} posições")
print(f"RMSE (Erro Quadrático Médio): {rmse:.2f} posições")

# Exemplo de comparação real vs previsto
comparison = pd.DataFrame({'Real': y_test, 'Previsto': y_pred}).head(10)
display(comparison)

## 6. Visualização
### Feature Importance

In [ ]:
plt.figure(figsize=(10, 8))
xgb.plot_importance(model, max_num_features=15, height=0.5, importance_type='weight', title='Top 15 Features (Weight)')
plt.show()

### Real vs Previsto

In [ ]:
plt.figure(figsize=(8, 8))
sns.scatterplot(x=y_test, y=y_pred, alpha=0.5)
plt.plot([1, 20], [1, 20], color='red', linestyle='--') # Linha ideal
plt.xlabel("Posição Real")
plt.ylabel("Posição Prevista")
plt.title("Real vs Previsto")
plt.xlim(0, 21)
plt.ylim(0, 21)
plt.show()

## 7. Salvar Modelo

In [ ]:
model_filename = 'xgboost_f1_model.json'
model.save_model(model_filename)
print(f"Modelo salvo como {model_filename}")

# Opcional: Upload para S3
# s3 = boto3.client('s3')
# s3.upload_file(model_filename, BUCKET, f'models/{model_filename}')